# Stage 2 Notebook 53 - Exp2XX K=64 query head + DAB anchor + DN denoising + VFL

**Stabilized version of NB47.** NB47 (K=64 query + Hungarian + VFL) was the cls breakthrough but its gap PEAKED at epoch 9 (0.099) then degraded to 0.063 by epoch 20 -- classic DETR-style query oscillation. NB50 (same head + full 70K data) collapsed completely (matched_iou=0.013) -- the queries couldn't ground themselves in the wider data distribution.

Both failure modes are addressed by DAB-DETR + DN-DETR techniques:

- **DAB anchors**: each query has a learnable (start_y, start_x, theta) anchor parameter that biases its spatial attention. Queries can't all converge to the same point. Stabilizes the geometry.
- **Denoising queries** (DN-DETR): during training, also pass `dn_num_groups=4` noised copies of every GT lane through the decoder. Each noised query must reconstruct its noiseless GT, which provides DENSE gradient signal that prevents query collapse. Standard cls/regr losses on the matched queries still apply.

`LaneQueryHeadAnchorDN` already implements both. Combined with VFL on matched_existence + Hungarian 1-to-1, this is the modern DETR recipe applied to lanes.

Reference: Liu et al. 'DAB-DETR: Dynamic Anchor Boxes are Better Queries for DETR' (ICLR 2022); Li et al. 'DN-DETR: Accelerate DETR Training by Introducing Query DeNoising' (CVPR 2022).

### Run mode
1. `DEBUG_MODE = True` smoke first.
2. `DEBUG_MODE = False` for 20 epochs limit=3000.
3. ~30-35 min wall-clock with AMP (DN queries add ~10% per step).

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp48_rmt_gca_query_anchor_dn_vfl_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp48_rmt_gca_query_anchor_dn_vfl_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp48_rmt_gca_query_anchor_dn_vfl_joint_smoke.log
OK exp48_rmt_gca_query_anchor_dn_vfl_joint.yaml
  lane_shape=(1, 64, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.5412 det_loss=3.2853 grad_cos=0.0829 lambda_lane=0.0696
  gate_stats={'gate/det_mean': 0.4982585906982422, 'gate/lane_mean': 0.4977600574493408, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp48_rmt_gca_query_anchor_dn_vfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp48_rmt_gca_query_anchor_dn_vfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp48_rmt_gca_query_anchor_dn_vfl_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp48_rmt_gca_query_anchor_dn_vfl_joint_short20.tar --epochs 20 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp48_rmt_gca_query_anchor_dn_vfl_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp48_rmt_gca_query_anchor_dn_vfl_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp48_rmt_gca_query_anchor_dn_vfl_joint.yaml --curve-tar /content/

0

## What to watch in Exp2XX training

Reference NB47 (K=64 query plain): peak gap 0.099 at ep9, decay to 0.063 at ep20. matched_iou=0.27.

Pass criteria at epoch 20:
- **gap doesn't decay**: pos-neg gap STABLE >= 0.08 from epoch 10 onwards (vs NB47's degradation).
- **matched_iou >= 0.35** -- DAB anchors give queries spatial bias they previously had to discover.
- **val/lane_f1 >= 0.30** and **val/lane_best_f1 >= 0.40** -- both above NB47.
- **val/lane/decoded_f1 >= 0.10** (5x NB47).

Failure signals:
- gap < 0.05: DAB anchors didn't help; query head fundamentally limited at K=64.
- matched_iou < 0.20: DN queries are dragging the geometry. Lower `dn_num_groups` to 2.
- decoded_f1 ~ NB47: stable plateau but no improvement. Pivot to anchor-head topk_fixed (Exp2WW).